In [ ]:
#Cách 1 clone: !git clone https://github.com/HoangPhuc-edward/ai-legal-graph-pipeline.git
#Cách 2 unrar: !unrar x "/content/ai-legal-retrieval-benchmark.rar" "/content/"

fatal: destination path 'ai-legal-graph-pipeline' already exists and is not an empty directory.


In [ ]:
%cd /content/ai-legal-retrieval-benchmark

/content/ai-legal-retrieval-benchmark/ai-legal-retrieval-benchmark


In [ ]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 2.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of deepeval to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of deepeval to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 4.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of google-cloud-vectorsearch to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 74.2

In [ ]:
!pip install -q langchain deepeval --upgrade
!pip install rank_bm25

In [ ]:
%%writefile .env
#EXAMPLE
# Neo4j AuraDB
NEO4J_URI=neo4j+s://xxxx.databases.neo4j.io
NEO4J_USER=xxxx
NEO4J_PASSWORD=xxxxx
# Google Cloud / Vertex AI
GCP_PROJECT=ID_PROJECT
GCP_LOCATION=global

# Hugging Face dataset
HF_DATASET_REPO=th1nhng0/vietnamese-legal-documents
# Model names (override here if a model id changes/becomes unavailable)
EMBEDDING_MODEL=gemini-embedding-001
LLM_MODEL_HEAVY=gemini-3.5-flash
LLM_MODEL_LIGHT=gemini-2.5-flash
GOOGLE_API_KEY=KEY_API_GOOGLE
# Pipeline intermediate file directory
DATA_DIR=./data


Overwriting .env


In [ ]:
!gcloud auth application-default login

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=l25OIeJUG4tr0hzC451ag1tYG1tSCW&prompt=consent&token_usage=remote&access_type=offline&code_challenge=UwnJzbxAYCXEq_b-DAP31RGOxOEGoVE5SiHX6nA5D4k&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0AdkVLPwJDuswlPTwC1kDSEweJk41CQ5S0hZ57__o_8A77hvPK8E_flMQPlBdaBKKle8Fgg

Credentials saved to file: [/content/.config/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).
Ca

In [ ]:
!gcloud auth application-default set-quota-project ID_PROJECT
!gcloud config set project ID_PROJECT

^C
^C


#MAKE SAMPLE QUESTION FROM JSONL

In [ ]:
import json
import random


def sample_questions_from_jsonl(
    input_file: str,
    output_file: str = "questions.jsonl",
    n: int = 100,
    seed: int = 42,
):
    """
    Lấy ngẫu nhiên n mẫu từ file JSONL và xuất ra JSONL:
    {"question": "<title>\n\n<body>"}
    """

    with open(input_file, "r", encoding="utf-8") as f:
        data = [json.loads(line) for line in f if line.strip()]

    random.seed(seed)
    samples = random.sample(data, min(n, len(data)))

    with open(output_file, "w", encoding="utf-8") as f:
        for item in samples:
            record = {
                "question": f"{item['title'].strip()}\n\n{item['body'].strip()}"
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"Saved {len(samples)} questions to {output_file}")
sample_questions_from_jsonl(
    input_file="data/filtered_tax_biz.jsonl",
    output_file="data/questions.jsonl",
    n=24
)

#RETRIEVAL

In [ ]:
!python -c "from retrieval.bm25_index import build_index; build_index()" #Để tạo index cho BM25

In [ ]:
from retrieval.pipeline import run_pipeline

In [ ]:
#Example 1
result= run_pipeline("HD thuê nhà thời hạn 1 năm không công chứng\n\nKính thưa Luật sư! Nhờ luật sư và các cộng sự tư vấn giúp e trường hợp này: Công ty em mới thành lập, khi nộp lên Cơ quan Thuế quận nơi mà trụ sở Cty e đóng thì phải có Hợp đồng thuê nhà (đối với doanh nghiệp k có nhà). Tuy nhiên, Hơp đồng đó có thời hạn 1 năm mà không công chứng. Theo e đọc tr BLDS Điều 492. Hình thức hợp đồng thuê nhà ở Hợp đồng thuê nhà ở phải được lập thành văn bản, nếu thời hạn thuê từ sáu tháng trở lên thì phải có công chứng hoặc chứng thực và phải đăng ký, trừ trường hợp pháp luật có quy định khác.\" Vậy, cho e hỏi và tư vấn giúp e trường hợp này sao cho đúng luật: Hợp đồng e chưa công chứng thì làm cách nào cho đúng lại ạ? cơ sở pháp lý nào e được thay đổi nội dung đó Em có phải soạn Phụ lục HĐ hay nộp HĐ ký mới có công chứng ạ? Hơn nữa khi khấu trừ thuế hình như HĐ thuê này phải đóng các loại phí , thuế gì mới đủ điều kiện và hợp lệ- đúng k ạ? E xin chân thành cảm ơn", mode="hybrid")
print(result['answer'])

Chào bạn, dưới đây là ý kiến tư vấn pháp lý dựa trên câu hỏi của bạn và các quy định pháp luật Việt Nam hiện hành:

### 1. Về phạm vi áp dụng và hiệu lực của tài liệu cung cấp (Context)
*   **Về văn bản trong Context:** Văn bản được cung cấp trong phần context là **Thông tư số 04/2005/TT-BKH** hướng dẫn về thành lập, tổ chức lại, đăng ký kinh doanh và giải thể *công ty nhà nước*. 
*   **Giới hạn của Context:** Nội dung câu hỏi của bạn liên quan đến **Hợp đồng thuê nhà làm trụ sở của doanh nghiệp tư nhân/thường** và **pháp luật về Thuế, Dân sự**. Do đó, tài liệu trong context hoàn toàn không chứa thông tin để giải quyết câu hỏi của bạn. 
*   **Cơ sở giải quyết:** Dưới đây là câu trả lời dựa trên hệ thống pháp luật Việt Nam hiện hành (ngoài context) để hỗ trợ bạn một cách chính xác nhất.

---

### 2. Giải quyết vấn đề: Hợp đồng thuê nhà 1 năm không công chứng

#### a. Hợp đồng thuê nhà 1 năm không công chứng có hợp pháp không?
*   **Cơ sở pháp lý cũ:** Điều 492 Bộ luật Dân sự 2005 (mà bạ

In [ ]:
import pandas as pd
#RUNNING FOR QUESTIONS.jsonl
questions = pd.read_json(
    "/content/ai-legal-retrieval-benchmark/ai-legal-retrieval-benchmark/data/questions.jsonl",#Link của jsonl: JSONL có dạng [{"question":"...."},{"question":"...."},...]
    lines=True
)

results = []

for _, row in questions.iterrows():
    result = run_pipeline(
        row["question"],
        mode="vector"
    )
    results.append(result)

print(len(results))

/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/content/ai-legal-retrieval-benchmark/ai-legal-retrieval-benchmark/retrieval/prompts.py:32: LangChainDeprecationWarning: The class `ChatVertexAI` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import ChatGoogleGenerativeAI``.
  _llm_answer = ChatVertexAI(
/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:114: UserWarning: Your applicati

4


In [ ]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

questions = pd.read_json(
    "/content/ai-legal-retrieval-benchmark/ai-legal-retrieval-benchmark/data/questions.jsonl",
    lines=True,
)

modes = ["vector", "bm25", "hybrid", "graphrag"]


def run_one(idx, question, mode):
    result = run_pipeline(question, mode=mode)
    return {
        "id": idx,
        "mode": mode,
        "question": question,
        "result": result,
    }


results = []

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [
        executor.submit(run_one, idx, row["question"], mode)
        for mode in modes
        for idx, row in questions.iterrows()
    ]

    for future in as_completed(futures):
        results.append(future.result())

# Sắp xếp lại theo mode rồi id
mode_order = {mode: i for i, mode in enumerate(modes)}
results.sort(key=lambda x: (mode_order[x["mode"]], x["id"]))

print(f"Total results: {len(results)}")

In [ ]:
results

[{'question': 'HD thuê nhà thời hạn 1 năm không công chứng\n\nKính thưa Luật sư! Nhờ luật sư và các cộng sự tư vấn giúp e trường hợp này: Công ty em mới thành lập, khi nộp lên Cơ quan Thuế quận nơi mà trụ sở Cty e đóng thì phải có Hợp đồng thuê nhà (đối với doanh nghiệp k có nhà). Tuy nhiên, Hơp đồng đó có thời hạn 1 năm mà không công chứng. Theo e đọc tr BLDS Điều 492. Hình thức hợp đồng thuê nhà ở Hợp đồng thuê nhà ở phải được lập thành văn bản, nếu thời hạn thuê từ sáu tháng trở lên thì phải có công chứng hoặc chứng thực và phải đăng ký, trừ trường hợp pháp luật có quy định khác." Vậy, cho e hỏi và tư vấn giúp e trường hợp này sao cho đúng luật: Hợp đồng e chưa công chứng thì làm cách nào cho đúng lại ạ? cơ sở pháp lý nào e được thay đổi nội dung đó Em có phải soạn Phụ lục HĐ hay nộp HĐ ký mới có công chứng ạ? Hơn nữa khi khấu trừ thuế hình như HĐ thuê này phải đóng các loại phí , thuế gì mới đủ điều kiện và hợp lệ- đúng k ạ? E xin chân thành cảm ơn,',
  'mode': 'vector',
  'answer'

#BENCHMARK

In [5]:
# BENCHMARK
!python -m benchmark.run_benchmark --eval-set data/questions.jsonl --modes vector   --max-workers 4